Initial Setup File for 1st setup

In [1]:
# Check GPU is running or not
import torch
print(torch.cuda.is_available())

False


In [ ]:
# Clone Github Repo & Verify it again
!git clone https://github.com/IITAkshayKaleBuilds/Retrieval-Augmented-Generation-LangGraph-Ollama

# check the contents of the cloned repo
%cd Retrieval-Augmented-Generation-LangGraph-Ollama

# check the contents of the RAG_Applications folder
!ls

import sys
sys.path.insert(0, '/content/Retrieval-Augmented-Generation-LangGraph-Ollama')

Cloning into 'Retrieval-Augmented-Generation-LangGraph-Ollama'...
remote: Enumerating objects: 249, done.
remote: Counting objects: 100% (249/249), done.
remote: Compressing objects: 100% (179/179), done.
remote: Total 249 (delta 146), reused 158 (delta 66), pack-reused 0 (from 0)
Receiving objects: 100% (249/249), 12.07 MiB | 37.56 MiB/s, done.
Resolving deltas: 100% (146/146), done.


In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Set Up Config: 
import os
from google.colab import userdata

os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")

if os.environ.get("LANGSMITH_API_KEY"):
  print("LANGSMITH_API_KEY FOUND")
else:
  print("API KEY NOT FOUND")

# Non-sensitive configs
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com/"
os.environ["LANGCHAIN_PROJECT"] = "Retrieval-Augmented-Generation-LangGraph-Ollama"


LANGSMITH_API_KEY FOUND


In [ ]:
# Steps to install ollama in runtime of colab
!apt-get update && apt-get install -y zstd

# Install Ollama in Colab
!curl -fsSL https://ollama.com/install.sh | sh

# start Ollama
!nohup ollama serve > ollama.log 2>&1 &


Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,475 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,955 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,842 kB]
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubu

In [ ]:
# Pull Ollama Model:
!ollama pull mistral:7b

# Pull Embedding Model
!ollama pull embeddinggemma:300m

# Verify Model
!ollama list

In [ ]:
# Install Project Requirements
!pip install -q -r requirements.txt

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 29.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 10.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 19.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 92.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.7/506.7 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 83.7 MB/s eta 0:00:00

In [ ]:
# Final Check
import langchain
print("OK")

OK


In [ ]:
# Advanced RAG - Data Ingestion Pipeline

import os
import hashlib
from pathlib import Path

from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_core.documents import Document

from docling.document_converter import DocumentConverter

In [ ]:
# configurations
DATA_DIR = "/content/Retrieval-Augmented-Generation-LangGraph-Ollama/RAG_Applications/data"
CHROMA_DIR = "/content/drive/MyDrive/chroma_financial_db"
COLLECTION_NAME = "financial_docs"
EMBEDDING_MODEL = 'embeddinggemma:300m'
BASE_URL = 'http://127.0.0.1:11434'
LLM_MODEL = "mistral:7b"
DEBUG_PATH = "/content/drive/MyDrive/debug_logs"

In [ ]:
# Create Chroma Vector Store with Ollama Embeddings
embeddings= OllamaEmbeddings(model=EMBEDDING_MODEL, base_url=BASE_URL, num_ctx=8192)

vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR
)

In [ ]:
# Data Ingestion Pipeline:
from RAG_Applications.scripts import data_ingestion

data_path = Path(DATA_DIR)
pdf_files = list(data_path.rglob("*.pdf"))

for pdf_path in pdf_files:
    data_ingestion.ingest_docs_in_vectordb(pdf_path)

print("Total documents in DB:", vector_store._collection.count())  
#
vector_store._collection.count()


In [ ]:
# track the processed files
existing_docs = vector_store.get(where={"file_hash": {"$ne": ""}}, include=['metadatas'])
processed_hashes = [m.get('file_hash') for m in existing_docs['metadatas'] if m.get('file_hash')]
processed_hashes = set(processed_hashes)
existing_docs

In [ ]:
#
results = vector_store.search("What is Tesla's revenue for Q1 2024", search_type="similarity")

results